In [ ]:
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestRegressor
from sklearn.neighbors import KNeighborsRegressor
from sklearn.linear_model import BayesianRidge
from sklearn.metrics import mean_squared_error
# import yfinance as yf
import matplotlib.pyplot as plt
import math

from statsmodels.tsa.stattools import adfuller, kpss
from statsmodels.graphics.tsaplots import plot_acf

from statsmodels.tsa.regime_switching.markov_regression import MarkovRegression
import warnings
import itertools

warnings.filterwarnings('ignore')

In [2]:
"""
Data Preprocessing Module
Handles loading, normalization, and dataset creation for time series forecasting
"""

import numpy as np
import pandas as pd
import torch
from torch.utils.data import TensorDataset, DataLoader


class TimeSeriesDataset:
    """Manages time series data loading and preprocessing"""

    def __init__(self, csv_path, target_columns=None, exclude_columns=None):
        """
        Initialize dataset from CSV file

        Args:
            csv_path: Path to CSV file
            target_columns: List of column names to use (None = all numeric columns)
            exclude_columns: List of column names to exclude (applied after target_columns)
        """
        self.csv_path = csv_path
        self.data = pd.read_csv(csv_path)

        # Select target columns
        if target_columns is not None:
            self.data = self.data[target_columns]
        else:
            # Use all columns except the first (assumed to be timestamp/index)
            self.data = self.data[self.data.columns[1:]]

        # Exclude specific columns if specified
        if exclude_columns is not None:
            self.data = self.data.drop(columns=exclude_columns, errors='ignore')

        self.num_variables = self.data.shape[1]
        self.total_timesteps = self.data.shape[0]

        print(f"Loaded dataset: {self.total_timesteps} timesteps, {self.num_variables} variables")

    def get_data(self):
        """Return the processed dataframe"""
        return self.data

    def get_numpy(self):
        """Return data as numpy array"""
        return self.data.to_numpy().astype(np.float32)


def create_dataloaders(data, seq_len, forecast_len, batch_size=64,
                       train_ratio=0.7, valid_ratio=0.1, step=1):
    """
    Create train, validation, and test dataloaders

    Args:
        data: numpy array of shape (T_total, M)
        seq_len: Length of input sequence (lookback window)
        forecast_len: Length of forecast horizon
        batch_size: Batch size for training
        train_ratio: Proportion of data for training
        valid_ratio: Proportion of data for validation
        step: Step size for sliding window

    Returns:
        train_loader, valid_loader, test_loader
    """
    T_total = data.shape[0]
    train_end = int(train_ratio * T_total)
    valid_end = int((train_ratio + valid_ratio) * T_total)
    scaler = StandardScaler()
    scaler.fit(data[:train_end])
    data_scaled = scaler.transform(data).astype(np.float32)

    print(f"\nData Split:")
    print(f"  Total timesteps: {T_total}")
    print(f"  Train: 0 to {train_end} ({train_ratio*100:.0f}%)")
    print(f"  Valid: {train_end} to {valid_end} ({valid_ratio*100:.0f}%)")
    print(f"  Test: {valid_end} to {T_total} ({(1-train_ratio-valid_ratio)*100:.0f}%)")

    train_data = data_scaled[:train_end]
    valid_data = data_scaled[train_end:valid_end]
    test_data = data_scaled[valid_end:]

    def sliding_window(d):
        """Create sliding windows from time series data"""
        X, y = [], []
        total = seq_len + forecast_len
        for i in range(0, len(d) - total + 1, step):
            X.append(d[i:i + seq_len])
            y.append(d[i + seq_len:i + total])
        return (torch.from_numpy(np.array(X)).float(),
                torch.from_numpy(np.array(y)).float())

    print(len(train_data),len(valid_data))
    X_train, y_train = sliding_window(train_data)
    X_valid, y_valid = sliding_window(valid_data)
    X_test, y_test = sliding_window(test_data)

    print(f"\nDataloader Shapes:")
    print(f"  Train: X={X_train.shape}, y={y_train.shape}")
    print(f"  Valid: X={X_valid.shape}, y={y_valid.shape}")
    print(f"  Test:  X={X_test.shape}, y={y_test.shape}")

    train_loader = DataLoader(
        TensorDataset(X_train, y_train),
        batch_size=batch_size,
        shuffle=True
    )
    valid_loader = DataLoader(
        TensorDataset(X_valid, y_valid),
        batch_size=batch_size,
        shuffle=False
    )
    test_loader = DataLoader(
        TensorDataset(X_test, y_test),
        batch_size=batch_size,
        shuffle=False
    )

    return train_loader, valid_loader, test_loader


def normalize_data(data):
    """
    Z-score normalization across time dimension

    Args:
        data: pandas DataFrame or numpy array

    Returns:
        normalized_data, mean, std
    """
    if isinstance(data, pd.DataFrame):
        data = data.to_numpy().astype(np.float32)

    mean = data.mean(axis=0, keepdims=True)
    std = data.std(axis=0, keepdims=True)
    data_norm = (data - mean) / (std + 1e-8)

    return data_norm


def denormalize_data(data_norm, mean, std):
    """
    Reverse z-score normalization

    Args:
        data_norm: Normalized data
        mean: Original mean
        std: Original std

    Returns:
        denormalized data
    """
    if isinstance(data_norm, pd.DataFrame):
        data_norm = data_norm.to_numpy().astype(np.float32)

    return data_norm * (std + 1e-8) + mean

In [3]:
"""
Training and Evaluation Module
Handles model training, validation, and testing
"""

import torch
import torch.nn as nn
import numpy as np
from tqdm import tqdm


class Trainer:
    """Handles model training and evaluation"""

    def __init__(self, model, device='cpu'):
        """
        Initialize trainer

        Args:
            model: PatchTST model instance
            device: Device to train on ('cpu' or 'cuda')
        """
        self.model = model.to(device)
        self.device = device
        self.criterion = nn.MSELoss()
        self.history = {
            'train_loss': [],
            'valid_loss': [],
            'test_loss': None
        }

    def train_epoch(self, train_loader, optimizer, accumulation_steps=1):
        """Train for one epoch"""
        self.model.train()
        epoch_loss = 0.0
        optimizer.zero_grad()

        pbar = tqdm(enumerate(train_loader), total=len(train_loader), desc="Training")
        for i, (x_batch, y_batch) in pbar:
            x_batch = x_batch.to(self.device)
            y_batch = y_batch.permute(0, 2, 1).to(self.device)

            out = self.model(x_batch)
            loss = self.criterion(out, y_batch) / accumulation_steps
            loss.backward()

            if (i + 1) % accumulation_steps == 0:
                optimizer.step()
                optimizer.zero_grad()

            epoch_loss += loss.item() * accumulation_steps
            pbar.set_postfix({'loss': f'{loss.item() * accumulation_steps:.4f}'})

        # Handle remaining batches
        if (i + 1) % accumulation_steps != 0:
            optimizer.step()
            optimizer.zero_grad()

        return epoch_loss / len(train_loader)

    def validate(self, valid_loader):
        """Validate the model"""
        self.model.eval()
        valid_loss = 0.0

        with torch.no_grad():
            for x_batch, y_batch in valid_loader:
                x_batch = x_batch.to(self.device)
                y_batch = y_batch.permute(0, 2, 1).to(self.device)
                out = self.model(x_batch)
                valid_loss += self.criterion(out, y_batch).item()

        return valid_loss / len(valid_loader)

    def train(self, train_loader, valid_loader, epochs=10, lr=1e-4,
              accumulation_steps=1, patience=5):
        """
        Train the model

        Args:
            train_loader: Training data loader
            valid_loader: Validation data loader
            epochs: Number of training epochs
            lr: Learning rate
            accumulation_steps: Gradient accumulation steps (simulates larger batch size)
            patience: Early stopping patience

        Returns:
            Training history dictionary
        """
        optimizer = torch.optim.Adam(self.model.parameters(), lr=lr)
        # torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)
        scheduler = torch.optim.lr_scheduler.OneCycleLR(optimizer = optimizer,
                                            steps_per_epoch = len(train_loader),
                                            pct_start = 0.3,
                                            epochs = epochs,
                                            max_lr = lr)

        best_valid_loss = float('inf')
        patience_counter = 0

        print(f"\nTraining Configuration:")
        print(f"  Epochs: {epochs}")
        print(f"  Learning rate: {lr}")
        print(f"  Accumulation steps: {accumulation_steps}")
        print(f"  Device: {self.device}")
        print(f"  Early stopping patience: {patience}")

        for epoch in range(epochs):
            print(f"\n{'='*60}")
            print(f"Epoch {epoch+1}/{epochs}")
            print(f"{'='*60}")

            # Training
            train_loss = self.train_epoch(train_loader, optimizer, accumulation_steps)

            # Validation
            valid_loss = self.validate(valid_loader)

            # Learning rate scheduling
            scheduler.step()

            # Record history
            self.history['train_loss'].append(train_loss)
            self.history['valid_loss'].append(valid_loss)

            print(f"\nEpoch {epoch+1} Summary:")
            print(f"  Train Loss: {train_loss:.4f}")
            print(f"  Valid Loss: {valid_loss:.4f}")
            print(f"  Learning Rate: {scheduler.get_last_lr()[0]:.6f}")

            # Early stopping
            if valid_loss < best_valid_loss:
                best_valid_loss = valid_loss
                patience_counter = 0
                # Save best model
                torch.save(self.model.state_dict(), '/best_model.pt')
                print(f"  ✓ New best model saved (Valid Loss: {best_valid_loss:.4f})")
            else:
                patience_counter += 1
                print(f"  No improvement ({patience_counter}/{patience})")

                if patience_counter >= patience:
                    print(f"\nEarly stopping triggered after {epoch+1} epochs")
                    break

        # Load best model
        self.model.load_state_dict(torch.load('/best_model.pt'))
        print(f"\n{'='*60}")
        print(f"Training completed. Best validation loss: {best_valid_loss:.4f}")
        print(f"{'='*60}")

        return self.history

    def evaluate(self, test_loader, return_predictions=False):
        """
        Evaluate model on test set

        Args:
            test_loader: Test data loader
            return_predictions: Whether to return predictions and targets

        Returns:
            test_loss (and optionally predictions, targets)
        """
        self.model.eval()
        test_loss = 0.0
        all_predictions = []
        all_targets = []

        print("\nEvaluating on test set...")
        with torch.no_grad():
            for x_batch, y_batch in tqdm(test_loader):
                x_batch = x_batch.to(self.device)
                y_batch = y_batch.permute(0, 2, 1).to(self.device)
                out = self.model(x_batch)
                test_loss += self.criterion(out, y_batch).item()

                if return_predictions:
                    all_predictions.append(out.cpu().numpy())
                    all_targets.append(y_batch.cpu().numpy())

        test_loss = test_loss / len(test_loader)
        self.history['test_loss'] = test_loss

        print(f"\n{'='*60}")
        print(f"Test Loss (MSE): {test_loss:.4f}")
        print(f"Test RMSE: {np.sqrt(test_loss):.4f}")
        print(f"{'='*60}")

        if return_predictions:
            predictions = np.concatenate(all_predictions, axis=0)
            targets = np.concatenate(all_targets, axis=0)
            return test_loss, predictions, targets

        return test_loss

    def get_history(self):
        """Return training history"""
        return self.history

# Model

In [4]:
import torch
import torch.nn as nn

class RevIN(nn.Module):
    """
    Reversible Instance Normalization — exact implementation from PatchTST.
    Source: github.com/yuqinie98/PatchTST/blob/main/PatchTST_supervised/layers/RevIN.py

    Expects input x: (B, seq_len, C)  [batch, time, channels]

    Usage:
        revin = RevIN(num_features=C, affine=True)
        x_norm = revin(x, 'norm')       # normalize before model
        y_denorm = revin(y_hat, 'denorm')  # reverse on output
    """
    def __init__(self, num_features: int, eps=1e-5, affine=True, subtract_last=False):
        super(RevIN, self).__init__()
        self.num_features   = num_features
        self.eps            = eps
        self.affine         = affine
        self.subtract_last  = subtract_last
        if self.affine:
            self.affine_weight = nn.Parameter(torch.ones(self.num_features))
            self.affine_bias   = nn.Parameter(torch.zeros(self.num_features))

    def forward(self, x, mode: str):
        if mode == 'norm':
            self._get_statistics(x)
            x = self._normalize(x)
        elif mode == 'denorm':
            x = self._denormalize(x)
        else:
            raise NotImplementedError(f"mode '{mode}' not supported")
        return x

    def _get_statistics(self, x):
        # reduce over time dim(s), keep channel dim intact
        dim2reduce = tuple(range(1, x.ndim - 1))    # (1,) for 3D input
        if self.subtract_last:
            self.last  = x[:, -1:, :]               # (B, 1, C)
        else:
            self.mean  = x.mean(dim=dim2reduce, keepdim=True).detach()   # (B, 1, C)
        self.stdev = torch.sqrt(
            x.var(dim=dim2reduce, keepdim=True, unbiased=False) + self.eps
        ).detach()

    def _normalize(self, x):
        x = x - (self.last if self.subtract_last else self.mean)
        x = x / self.stdev
        if self.affine:
            x = x * self.affine_weight + self.affine_bias
        return x

    def _denormalize(self, x):
        if self.affine:
            x = (x - self.affine_bias) / (self.affine_weight + self.eps ** 2)
        x = x * self.stdev
        x = x + (self.last if self.subtract_last else self.mean)
        return x

In [5]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
def compute_mae(model, test_loader, device):
    model.eval()
    mae_sum = count = 0
    with torch.no_grad():
        for x, y in test_loader:
            x = x.to(device)
            y = y.permute(0, 2, 1).to(device)
            mae_sum += F.l1_loss(model(x), y, reduction='sum').item()
            count   += y.numel()
    return mae_sum / count

class moving_avg(nn.Module):
    """
    Moving average block to highlight the trend of time series
    """
    def __init__(self, kernel_size, stride):
        super(moving_avg, self).__init__()
        print(stride)
        self.kernel_size = kernel_size
        self.avg = nn.AvgPool1d(kernel_size=kernel_size, stride=stride, padding=0)

    def forward(self, x):
        # padding on the both ends of time series
        front = x[:, 0:1, :].repeat(1, (self.kernel_size - 1) // 2, 1)
        end = x[:, -1:, :].repeat(1, (self.kernel_size - 1) // 2, 1)
        x = torch.cat([front, x, end], dim=1)
        x = self.avg(x.permute(0, 2, 1))
        x = x.permute(0, 2, 1)
        return x


class series_decomp(nn.Module):
    """
    Series decomposition block
    """
    def __init__(self, kernel_size):
        super(series_decomp, self).__init__()
        self.moving_avg = moving_avg(kernel_size, stride=1)

    def forward(self, x):
        moving_mean = self.moving_avg(x)
        res = x - moving_mean
        return res, moving_mean

class Model(nn.Module):
    """
    Decomposition-Linear
    """
    def __init__(self, configs):
        super(Model, self).__init__()
        self.seq_len = configs.seq_len
        self.pred_len = configs.forecast_len

        # Decompsition Kernel Size
        kernel_size = 25
        self.decompsition = series_decomp(kernel_size)
        self.individual = configs.individual
        self.channels = configs.enc_in

        if self.individual:
            self.Linear_Seasonal = nn.ModuleList()
            self.Linear_Trend = nn.ModuleList()

            for i in range(self.channels):
                self.Linear_Seasonal.append(nn.Linear(self.seq_len,self.pred_len))
                self.Linear_Trend.append(nn.Linear(self.seq_len,self.pred_len))

                # Use this two lines if you want to visualize the weights
                # self.Linear_Seasonal[i].weight = nn.Parameter((1/self.seq_len)*torch.ones([self.pred_len,self.seq_len]))
                # self.Linear_Trend[i].weight = nn.Parameter((1/self.seq_len)*torch.ones([self.pred_len,self.seq_len]))
        else:
            self.Linear_Seasonal = nn.Linear(self.seq_len,self.pred_len)
            self.Linear_Trend = nn.Linear(self.seq_len,self.pred_len)

            # Use this two lines if you want to visualize the weights
            # self.Linear_Seasonal.weight = nn.Parameter((1/self.seq_len)*torch.ones([self.pred_len,self.seq_len]))
            # self.Linear_Trend.weight = nn.Parameter((1/self.seq_len)*torch.ones([self.pred_len,self.seq_len]))

    def forward(self, x):
        # x: [Batch, Input length, Channel]
        B, L, M = x.shape
        x = self.revin.normalize(x)

        seasonal_init, trend_init = self.decompsition(x)
        seasonal_init, trend_init = seasonal_init.permute(0,2,1), trend_init.permute(0,2,1)
        if self.individual:
            seasonal_output = torch.zeros([seasonal_init.size(0),seasonal_init.size(1),self.pred_len],dtype=seasonal_init.dtype).to(seasonal_init.device)
            trend_output = torch.zeros([trend_init.size(0),trend_init.size(1),self.pred_len],dtype=trend_init.dtype).to(trend_init.device)
            for i in range(self.channels):
                seasonal_output[:,i,:] = self.Linear_Seasonal[i](seasonal_init[:,i,:])
                trend_output[:,i,:] = self.Linear_Trend[i](trend_init[:,i,:])
        else:
            seasonal_output = self.Linear_Seasonal(seasonal_init)
            trend_output = self.Linear_Trend(trend_init)

        x = seasonal_output + trend_output
        out = x.permute(0,2,1)
        out = self.revin.denormalize(out)
        return out # to [Batch, Output length, Channel]

In [6]:
class DLinearAdapted(nn.Module):
    """
    DLinear with output (B, C, pred_len) to match Trainer convention.
    Trainer permutes y_batch → (B, C, pred_len), so model must match.
    Original DLinear returns (B, pred_len, C) — the final permute is removed here.
    """
    def __init__(self, seq_len, forecast_len, enc_in, individual=False, kernel_size=25):
        super().__init__()
        self.seq_len    = seq_len
        self.pred_len   = forecast_len
        self.channels   = enc_in
        self.individual = individual
        self.decompsition = series_decomp(kernel_size)

        if self.individual:
            self.Linear_Seasonal = nn.ModuleList(
                [nn.Linear(seq_len, forecast_len) for _ in range(enc_in)])
            self.Linear_Trend = nn.ModuleList(
                [nn.Linear(seq_len, forecast_len) for _ in range(enc_in)])
        else:
            self.Linear_Seasonal = nn.Linear(seq_len, forecast_len)
            self.Linear_Trend    = nn.Linear(seq_len, forecast_len)

    def forward(self, x):
        # x: (B, seq_len, C)
        seasonal, trend = self.decompsition(x)
        seasonal = seasonal.permute(0, 2, 1)   # (B, C, seq_len)
        trend    = trend.permute(0, 2, 1)

        if self.individual:
            s_out = torch.zeros(seasonal.size(0), self.channels, self.pred_len,
                                dtype=seasonal.dtype, device=seasonal.device)
            t_out = torch.zeros_like(s_out)
            for i in range(self.channels):
                s_out[:, i, :] = self.Linear_Seasonal[i](seasonal[:, i, :])
                t_out[:, i, :] = self.Linear_Trend[i](trend[:, i, :])
        else:
            s_out = self.Linear_Seasonal(seasonal)   # (B, C, pred_len)
            t_out = self.Linear_Trend(trend)

        return s_out + t_out   # (B, C, pred_len)

In [7]:
class DLinearRevIN(nn.Module):
    """
    DLinear + RevIN, output (B, C, pred_len) to match your Trainer.

    RevIN flow (PatchTST convention):
        1. norm(x)          on input  (B, seq_len, C)
        2. model forward
        3. denorm(output)   on output (B, pred_len, C)  ← BEFORE final permute
        4. permute          → (B, C, pred_len)          ← for Trainer
    """
    def __init__(self, seq_len, forecast_len, enc_in,
                 individual=False, kernel_size=25,
                 revin=True, affine=True, subtract_last=False):
        super().__init__()
        self.seq_len   = seq_len
        self.pred_len  = forecast_len
        self.channels  = enc_in
        self.individual = individual
        self.revin     = revin

        if self.revin:
            self.revin_layer = RevIN(enc_in, affine=affine,
                                     subtract_last=subtract_last)

        self.decompsition = series_decomp(kernel_size)

        if self.individual:
            self.Linear_Seasonal = nn.ModuleList(
                [nn.Linear(seq_len, forecast_len) for _ in range(enc_in)])
            self.Linear_Trend = nn.ModuleList(
                [nn.Linear(seq_len, forecast_len) for _ in range(enc_in)])
        else:
            self.Linear_Seasonal = nn.Linear(seq_len, forecast_len)
            self.Linear_Trend    = nn.Linear(seq_len, forecast_len)

    def forward(self, x):
        # x: (B, seq_len, C)

        # ── 1. Instance Normalize ─────────────────────────────────
        if self.revin:
            x = self.revin_layer(x, 'norm')    # (B, seq_len, C), stats stored

        # ── 2. Decompose & Linear ─────────────────────────────────
        seasonal, trend = self.decompsition(x)
        seasonal = seasonal.permute(0, 2, 1)   # (B, C, seq_len)
        trend    = trend.permute(0, 2, 1)

        if self.individual:
            s_out = torch.zeros(x.size(0), self.channels, self.pred_len,
                                dtype=x.dtype, device=x.device)
            t_out = torch.zeros_like(s_out)
            for i in range(self.channels):
                s_out[:, i, :] = self.Linear_Seasonal[i](seasonal[:, i, :])
                t_out[:, i, :] = self.Linear_Trend[i](trend[:, i, :])
        else:
            s_out = self.Linear_Seasonal(seasonal)   # (B, C, pred_len)
            t_out = self.Linear_Trend(trend)

        out = (s_out + t_out).permute(0, 2, 1)        # (B, pred_len, C)

        # ── 3. Reverse Normalize ──────────────────────────────────
        if self.revin:
            out = self.revin_layer(out, 'denorm')      # (B, pred_len, C)

        return out.permute(0, 2, 1)                    # (B, C, pred_len) for Trainer

### Dlinear original without RevIn

In [8]:
import argparse
import os
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

# Reproducibility
SEED = 2021
torch.manual_seed(SEED)
np.random.seed(SEED)

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Device: {DEVICE}")

Device: cuda


In [ ]:

results = {}
from sklearn.preprocessing import StandardScaler
# ── Edit these ────────────────────────────────────────────────
DATA_PATH    = 'traffic.csv'
SEQ_LEN      = 336
PRED_LENS    = [720]
INDIVIDUAL   = True
BATCH_SIZE   = 8
ACCUM_STEPS  = 2
LR = 0.05
EPOCHS = 20
PATIENCE = 5
BEST_MODEL_PATH = './best_model.pt'

EXPECTED = {
    96:  (0.410, 0.282),
    192: (0.423, 0.287),
    336: (0.436, 0.296),
    720: (0.466, 0.315),
}


ts   = TimeSeriesDataset(DATA_PATH)
data = ts.get_numpy()          # (T, 21)
ENC_IN = ts.num_variables
print(f"Channels: {ENC_IN}  |  Total timesteps: {data.shape[0]}")


scaler = StandardScaler()
train_end = int(0.7 * len(data))
valid_end = int(0.8 * len(data))

scaler.fit(data[:train_end])
data_scaled = scaler.transform(data)   # apply globally

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
for pred_len in PRED_LENS:
    print(f"\n{'#'*60}")
    print(f"  DLinear | Traffic | L={SEQ_LEN} | T={pred_len}")
    print(f"{'#'*60}")

    # Dataloaders
    train_loader, valid_loader, test_loader = create_dataloaders(
        data         = data_scaled,
        seq_len      = SEQ_LEN,
        forecast_len = pred_len,
        batch_size   = BATCH_SIZE,
        train_ratio  = 0.7,
        valid_ratio  = 0.1,
        step         = 1,
    )

    # Model
    model = DLinearAdapted(
        seq_len      = SEQ_LEN,
        forecast_len = pred_len,
        enc_in       = ENC_IN,
        individual   = INDIVIDUAL,
        kernel_size  = 25,
    )

    # Patch save path so Trainer doesn't write to /
    trainer = Trainer(model, device=DEVICE)
    trainer.model
    trainer_module_path = BEST_MODEL_PATH

    # Train
    history = trainer.train(
        train_loader       = train_loader,
        valid_loader       = valid_loader,
        epochs             = EPOCHS,
        lr                 = LR,
        accumulation_steps = ACCUM_STEPS,
        patience           = PATIENCE,
    )

    # Evaluate
    test_mse = trainer.evaluate(test_loader)
    test_mae = compute_mae(model, test_loader, DEVICE)

    results[pred_len] = (test_mse, test_mae)
    e_mse, e_mae = EXPECTED[pred_len]
    print(f"\n  ▶ T={pred_len:3d}  MSE={test_mse:.4f} (expected {e_mse})  "
          f"MAE={test_mae:.4f} (expected {e_mae})")

In [ ]:

results = {}
from sklearn.preprocessing import StandardScaler
# ── Edit these ────────────────────────────────────────────────
DATA_PATH    = 'electricity.csv'
SEQ_LEN      = 336
PRED_LENS    = [720]
INDIVIDUAL   = True
BATCH_SIZE   = 8
ACCUM_STEPS  = 2
LR = 0.001
EPOCHS = 20
PATIENCE = 5
BEST_MODEL_PATH = './best_model.pt'

EXPECTED = {
    96:  (0.140, 0.237),
    192: (0.153, 0.249),
    336: (0.169, 0.267),
    720: (0.203, 0.301),
}

ts   = TimeSeriesDataset(DATA_PATH)
data = ts.get_numpy()          # (T, 21)
ENC_IN = ts.num_variables
print(f"Channels: {ENC_IN}  |  Total timesteps: {data.shape[0]}")


scaler = StandardScaler()
train_end = int(0.7 * len(data))
valid_end = int(0.8 * len(data))

scaler.fit(data[:train_end])
data_scaled = scaler.transform(data)   # apply globally

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
for pred_len in PRED_LENS:
    print(f"\n{'#'*60}")
    print(f"  DLinear | Electricity | L={SEQ_LEN} | T={pred_len}")
    print(f"{'#'*60}")

    # Dataloaders
    train_loader, valid_loader, test_loader = create_dataloaders(
        data         = data_scaled,
        seq_len      = SEQ_LEN,
        forecast_len = pred_len,
        batch_size   = BATCH_SIZE,
        train_ratio  = 0.7,
        valid_ratio  = 0.1,
        step         = 1,
    )

    # Model
    model = DLinearAdapted(
        seq_len      = SEQ_LEN,
        forecast_len = pred_len,
        enc_in       = ENC_IN,
        individual   = INDIVIDUAL,
        kernel_size  = 25,
    )

    # Patch save path so Trainer doesn't write to /
    trainer = Trainer(model, device=DEVICE)
    trainer.model
    trainer_module_path = BEST_MODEL_PATH

    # Train
    history = trainer.train(
        train_loader       = train_loader,
        valid_loader       = valid_loader,
        epochs             = EPOCHS,
        lr                 = LR,
        accumulation_steps = ACCUM_STEPS,
        patience           = PATIENCE,
    )

    # Evaluate
    test_mse = trainer.evaluate(test_loader)
    test_mae = compute_mae(model, test_loader, DEVICE)

    results[pred_len] = (test_mse, test_mae)
    e_mse, e_mae = EXPECTED[pred_len]
    print(f"\n  ▶ T={pred_len:3d}  MSE={test_mse:.4f} (expected {e_mse})  "
          f"MAE={test_mae:.4f} (expected {e_mae})")

Loaded dataset: 26304 timesteps, 321 variables
Channels: 321  |  Total timesteps: 26304

############################################################
  DLinear | Weather | L=336 | T=96
############################################################

Data Split:
  Total timesteps: 26304
  Train: 0 to 18412 (70%)
  Valid: 18412 to 21043 (10%)
  Test: 21043 to 26304 (20%)
18412 2631
